Q1. Read the data for January. How many columns are there?



In [11]:
import numpy as np
import pandas as pd

In [12]:
df = pd.read_parquet('yellow_tripdata_2021-01.parquet')
df.shape[1]

19

Q2. What's the standard deviation of the trips duration in January?



In [13]:
df.tpep_dropoff_datetime = pd.to_datetime(df.tpep_dropoff_datetime)
df.tpep_pickup_datetime = pd.to_datetime(df.tpep_pickup_datetime)

df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
np.std(df.duration)

np.float64(131.20056030282015)

Q3. What fraction of the records left after you dropped the outliers?


In [14]:
filtered_df_val = df[(df['duration'] >= 1) & (df['duration'] <= 60)]
percentage_remaining = (len(filtered_df_val) / len(df)) * 100
print(f'Percentage of data remaining after filtering: {percentage_remaining:.2f}%')


Percentage of data remaining after filtering: 98.06%


Q4. Turn the dataframe into a list of dictionaries (remember to re-cast the ids to strings - otherwise it will label encode them)
Fit a dictionary vectorizer
Get a feature matrix from it
What's the dimensionality of this matrix (number of columns)?

In [15]:
from sklearn.feature_extraction import DictVectorizer

categorical = ['PULocationID', 'DOLocationID', 'store_and_fwd_flag', 'payment_type']
numerical = ['trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 
             'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge']

for col in numerical:
    df[col].fillna(df[col].median(), inplace=True)
for col in categorical:
    df[col].fillna('missing', inplace=True)

df['PULocationID'] = df['PULocationID'].astype(str)
df['DOLocationID'] = df['DOLocationID'].astype(str)

val_dicts = df[categorical + numerical].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_val = dv.fit_transform(val_dicts)

print(f"Number of columns in feature matrix: {X_val.shape[1]}")


/var/folders/58/1jrwwbcj7mb8n97wtbklvw280000gn/T/ipykernel_36089/3892290957.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
/var/folders/58/1jrwwbcj7mb8n97wtbklvw280000gn/T/ipykernel_36089/3892290957.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always

Number of columns in feature matrix: 531


Q5. Train a plain linear regression model with default parameters, where duration is the response variable
Calculate the RMSE of the model on the training data
What's the RMSE on train?

In [28]:
from sklearn.linear_model import LinearRegression
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_squared_error

categorical = ['PULocationID', 'DOLocationID']
numerical = ['trip_distance', 'fare_amount', 'total_amount',]


subset_dicts = df_subset[categorical + numerical].to_dict(orient='records')
y_train_subset = df_subset['duration'].values 

dv = DictVectorizer(sparse=False)
X_train_subset = dv.fit_transform(subset_dicts)

lr = LinearRegression()
lr.fit(X_train_subset, y_train_subset)

y_train_pred_subset = lr.predict(X_train_subset)

rmse_train_subset = np.sqrt(mean_squared_error(y_train_subset, y_train_pred_subset))
print(f'RMSE on training subset: {rmse_train_subset:.2f}')


RMSE on training subset: 130.57


Now let's apply this model to the validation dataset (February 2023).

What's the RMSE on validation?

In [ ]:
from sklearn.metrics import mean_squared_error

df_val = pd.read_parquet('yellow_tripdata_2021-02.parquet')

df_val['PULocationID'] = df_val['PULocationID'].astype(str)
df_val['DOLocationID'] = df_val['DOLocationID'].astype(str)

for col in numerical:
    df_val[col].fillna(df_val[col].median(), inplace=True)

for col in categorical:
    df_val[col].fillna('missing', inplace=True)

if df_val.isnull().sum().sum() > 0:
    print("Warning: There are still missing values in the validation DataFrame.")

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts) 

y_val = df_val['duration'].values

y_val_pred = lr.predict(X_val)

rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred))
print(f'RMSE on validation data: {rmse_val:.2f}')


/var/folders/58/1jrwwbcj7mb8n97wtbklvw280000gn/T/ipykernel_36089/3291598880.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_val[col].fillna(df_val[col].median(), inplace=True)
/var/folders/58/1jrwwbcj7mb8n97wtbklvw280000gn/T/ipykernel_36089/3291598880.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting val

RMSE on validation data: 57.98
